In [ ]:
# ==============================================================================
# NOTEBOOK 01: FEATURE EXTRACTION (v4 - Targeted Balancing)
# ==============================================================================
# OBIETTIVO:
# 1. Pre-processare ogni file audio del dataset UrbanSound8K.
# 2. Applicare DATA AUGMENTATION MIRATA per portare le classi 'car_horn' e
#    'gun_shot' a un target di circa 1000 campioni ciascuna.
# 3. Estrarre un set completo di feature per le pipeline A, B, C, D.
# 4. Salvare le feature in file .npz, garantendo che i campioni aumentati
#    rimangano nel fold dell'originale.
#
# Questo notebook va eseguito una sola volta all'inizio del progetto.
# ==============================================================================

import os
import numpy as np
import pandas as pd
import librosa
from tqdm.notebook import tqdm
import warnings
import random

warnings.filterwarnings('ignore')

# --- 1. CONFIGURAZIONE DEI PERCORSI E PARAMETRI ---

# Percorsi principali
DATA_ROOT = "../data"
RAW_DATA_PATH = os.path.join(DATA_ROOT, "raw")
# Usa una cartella specifica per questo esperimento
PROCESSED_DATA_PATH = os.path.join(DATA_ROOT, "processed", "features_v2")
META_FILE_PATH = os.path.join(RAW_DATA_PATH, "UrbanSound8K.csv")

# Parametri audio
TARGET_SR = 22050
TARGET_LENGTH_S = 4

# Parametri di Data Augmentation
CLASSES_TO_AUGMENT = ['car_horn', 'gun_shot']
TARGET_SAMPLES_PER_CLASS = 1000
NOISE_LEVELS = [0.003, 0.006]
PITCH_STEPS = [-2, -1, 1, 2]
TIME_STRETCH_RATES = [0.9, 1.1]

print("Configurazione completata:")
print(f"  - Percorso dati processati: {PROCESSED_DATA_PATH}")
print(f"  - Classi da aumentare: {CLASSES_TO_AUGMENT}")

# --- 2. FUNZIONI DI DATA AUGMENTATION (COMBINABILI) ---

def augment_audio(y, sr):
    """
    Applica una combinazione casuale di tecniche di augmentation.
    """
    y_aug = y.copy()
    
    # Applica una o più trasformazioni casualmente
    transformations = [
        lambda y: y + random.choice(NOISE_LEVELS) * np.random.randn(len(y)),
        lambda y: librosa.effects.pitch_shift(y=y, sr=sr, n_steps=random.choice(PITCH_STEPS)),
        lambda y: librosa.effects.time_stretch(y=y, rate=random.choice(TIME_STRETCH_RATES))
    ]
    # Scegli a caso quante trasformazioni applicare (da 1 a 3)
    num_transforms_to_apply = random.randint(1, len(transformations))
    random.shuffle(transformations) # Applica in ordine casuale
    
    for i in range(num_transforms_to_apply):
        y_aug = transformations[i](y_aug)
        
    return np.clip(y_aug, -1.0, 1.0)

# --- 3. FUNZIONI DI PRE-PROCESSING E FEATURE EXTRACTION (INVARIATE) ---

# (Le funzioni load_and_standardize_audio, enhance_and_frame_audio, extract_all_features
# rimangono identiche a quelle della versione precedente, le ometto per brevità ma vanno incluse qui)
def load_and_standardize_audio(audio_path, target_sr):
    try:
        y, sr = librosa.load(audio_path, sr=None, mono=True)
        if sr != target_sr:
            y = librosa.resample(y=y, orig_sr=sr, target_sr=target_sr)
        return y, target_sr
    except Exception: return None, None
def enhance_and_frame_audio(y, target_sr, target_duration_s, trim_db=25):
    target_length_samples = int(target_sr * target_duration_s)
    y_trimmed, _ = librosa.effects.trim(y, top_db=trim_db)
    if len(y_trimmed) == 0: y_trimmed = y
    if len(y_trimmed) > target_length_samples:
        start_offset = (len(y_trimmed) - target_length_samples) // 2
        y_framed = y_trimmed[start_offset : start_offset + target_length_samples]
    else:
        pad_width = target_length_samples - len(y_trimmed)
        pad_left = pad_width // 2
        pad_right = pad_width - pad_left
        y_framed = np.pad(y_trimmed, (pad_left, pad_right), mode='constant')
    return y_framed
def extract_all_features(y, sr):
    features = {}
    features['log_mel_spec'] = librosa.power_to_db(librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128))
    mfcc_base = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40)
    features['mfcc'] = mfcc_base
    delta1 = librosa.feature.delta(mfcc_base)
    delta2 = librosa.feature.delta(mfcc_base, order=2)
    features['mfcc_delta_stack'] = np.stack([mfcc_base, delta1, delta2], axis=-1)
    features['chroma'] = librosa.feature.chroma_stft(y=y, sr=sr)
    features['contrast'] = librosa.feature.spectral_contrast(y=y, sr=sr)
    features['centroid'] = librosa.feature.spectral_centroid(y=y, sr=sr).flatten()
    features['rolloff'] = librosa.feature.spectral_rolloff(y=y, sr=sr).flatten()
    features['zcr'] = librosa.feature.zero_crossing_rate(y=y).flatten()
    return features


# --- 4. CALCOLO PRELIMINARE PER AUGMENTATION ---

print("\nAnalisi del dataset per pianificare l'oversampling mirato...")
df_meta = pd.read_csv(META_FILE_PATH)
class_counts = df_meta['class'].value_counts()
print("Distribuzione delle classi originale:")
print(class_counts)

# Calcola quanti campioni generare solo per le classi target
augmentation_plan = {}
for class_name in CLASSES_TO_AUGMENT:
    count = class_counts.get(class_name, 0)
    if count > 0 and count < TARGET_SAMPLES_PER_CLASS:
        samples_to_generate = TARGET_SAMPLES_PER_CLASS - count
        n_augmentations_per_file = samples_to_generate // count
        extra_augmentations = samples_to_generate % count
        augmentation_plan[class_name] = {
            'base_aug_count': n_augmentations_per_file,
            'extra_aug_count': extra_augmentations,
            'current_extra_count': 0  # Tracker per distribuire gli extra
        }
        print(f"Classe '{class_name}': mancano {samples_to_generate} campioni. "
              f"Si genereranno circa {n_augmentations_per_file+1} copie per file.")

# --- 5. WORKFLOW DI ESECUZIONE COMPLETO ---

print("\nAvvio del processo di estrazione e bilanciamento mirato...")
os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)

total_files_saved = 0
error_count = 0

for _, row in tqdm(df_meta.iterrows(), total=len(df_meta), desc="Processing & Balancing"):
    fold_num = row['fold']
    filename = row['slice_file_name']
    class_name = row['class']
    class_id = row['classID']
    
    src_path = os.path.join(RAW_DATA_PATH, f"fold{fold_num}", filename)
    dst_dir = os.path.join(PROCESSED_DATA_PATH, f"fold{fold_num}")
    os.makedirs(dst_dir, exist_ok=True)
    
    base_name = os.path.splitext(filename)[0]

    try:
        y_raw, sr = load_and_standardize_audio(src_path, TARGET_SR)
        if y_raw is None:
            error_count += 1
            continue
            
        # 1. Processa e salva sempre il file ORIGINALE
        y_processed = enhance_and_frame_audio(y_raw, sr, TARGET_LENGTH_S)
        features = extract_all_features(y_processed, sr)
        dst_path_original = os.path.join(dst_dir, f"{base_name}.npz")
        if not os.path.exists(dst_path_original):
            np.savez_compressed(dst_path_original, class_id=class_id, fold=fold_num, **features)
            total_files_saved += 1
            
        # 2. Applica AUGMENTATION solo se la classe è nella lista target
        if class_name in augmentation_plan:
            plan = augmentation_plan[class_name]
            num_augmentations = plan['base_aug_count']
            
            # Aggiungi una copia extra se necessario, distribuendo uniformemente
            if plan['current_extra_count'] < plan['extra_aug_count']:
                num_augmentations += 1
                plan['current_extra_count'] += 1
                
            for i in range(num_augmentations):
                y_augmented = augment_audio(y_raw, sr)
                y_processed_aug = enhance_and_frame_audio(y_augmented, sr, TARGET_LENGTH_S)
                features_aug = extract_all_features(y_processed_aug, sr)
                
                aug_name = f"{base_name}_aug_{i}"
                dst_path_aug = os.path.join(dst_dir, f"{aug_name}.npz")
                
                if not os.path.exists(dst_path_aug):
                    np.savez_compressed(dst_path_aug, class_id=class_id, fold=fold_num, **features_aug)
                    total_files_saved += 1

    except Exception as e:
        print(f"Errore irreversibile processando {filename}: {e}")
        error_count += 1
        
print("\n--- Processo Completato ---")
print(f"File .npz totali salvati: {total_files_saved}")
print(f"Errori riscontrati: {error_count}")
print(f"I dati sono pronti in: {PROCESSED_DATA_PATH}")

Configurazione completata:
  - Percorso dati processati: ../data\processed\features_v4_targeted_balance
  - Classi da aumentare: ['car_horn', 'gun_shot']

Analisi del dataset per pianificare l'oversampling mirato...
Distribuzione delle classi originale:
class
dog_bark            1000
children_playing    1000
air_conditioner     1000
street_music        1000
jackhammer          1000
engine_idling       1000
drilling            1000
siren                929
car_horn             429
gun_shot             374
Name: count, dtype: int64
Classe 'car_horn': mancano 571 campioni. Si genereranno circa 2 copie per file.
Classe 'gun_shot': mancano 626 campioni. Si genereranno circa 2 copie per file.

Avvio del processo di estrazione e bilanciamento mirato...


Processing & Balancing:   0%|          | 0/8732 [00:00<?, ?it/s]


--- Processo Completato ---
File .npz totali salvati: 9929
Errori riscontrati: 0
I dati sono pronti in: ../data\processed\features_v4_targeted_balance
